# 03. 자산 간 상관관계 분석 (Cross-Asset Correlation)

서로 다른 자산 클래스(암호화폐, 주식, 금) 간의 수익률 상관관계를 분석한다.

**핵심 질문**:
- BTC와 금은 실제로 상관관계가 있는가?
- BTC와 기술주(NVDA, AAPL)는 함께 움직이는가?
- 삼성전자와 SK하이닉스는 얼마나 밀접한가?
- 상관관계는 시간에 따라 어떻게 변하는가?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

from db import load_prices, get_asset_list

plt.rcParams['figure.figsize'] = (14, 10)
sns.set_theme(style='whitegrid')

In [ ]:
assets = get_asset_list()
assets

## 1. 데이터 정렬 (공통 타임스탬프)

In [ ]:
# 모든 자산을 시간 단위로 리샘플링하고 수익률 계산
RESAMPLE_FREQ = '1h'

returns_dict = {}
prices_dict = {}

for _, row in assets.iterrows():
    source, symbol = row['source'], row['symbol']
    label = f"{symbol}\n({source})"

    df = load_prices(symbol=symbol, source=source)
    if df.empty:
        continue

    # 리샘플링
    resampled = df.set_index('collected_at')['price'].resample(RESAMPLE_FREQ).last().ffill()
    prices_dict[label] = resampled

    # 수익률
    returns_dict[label] = resampled.pct_change().dropna()

# 수익률 DataFrame (공통 인덱스)
returns_df = pd.DataFrame(returns_dict).dropna()
prices_df = pd.DataFrame(prices_dict).dropna()

print(f"공통 타임스탬프 수: {len(returns_df)}")
print(f"기간: {returns_df.index.min()} ~ {returns_df.index.max()}")
print(f"\n자산: {list(returns_df.columns)}")

## 2. 피어슨 상관관계 행렬

In [ ]:
# 상관관계 행렬 계산
corr_matrix = returns_df.corr()

# 클러스터맵
g = sns.clustermap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r',
                   center=0, vmin=-1, vmax=1,
                   linewidths=1, figsize=(12, 10),
                   dendrogram_ratio=0.15)
g.fig.suptitle('자산 수익률 상관관계 행렬 (피어슨)', y=1.02, fontsize=14)
plt.show()

In [ ]:
# 상관관계 쌍별 정렬 (가장 높은/낮은 상관관계)
pairs = []
cols = corr_matrix.columns
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        pairs.append({
            '자산 A': cols[i],
            '자산 B': cols[j],
            '상관계수': corr_matrix.iloc[i, j]
        })

pairs_df = pd.DataFrame(pairs).sort_values('상관계수', ascending=False)
print("=== 상관관계 순위 (높은 순) ===")
pairs_df

## 3. 롤링 상관관계

In [ ]:
# 관심 쌍 정의
INTEREST_PAIRS = []

# 자동으로 쌍 생성: 모든 조합 중 상위 3개 + 하위 3개
if len(pairs_df) >= 6:
    top_pairs = pairs_df.head(3)
    bottom_pairs = pairs_df.tail(3)
    selected = pd.concat([top_pairs, bottom_pairs])
else:
    selected = pairs_df

for _, p in selected.iterrows():
    INTEREST_PAIRS.append((p['자산 A'], p['자산 B']))

print(f"분석할 쌍: {INTEREST_PAIRS}")

In [ ]:
ROLLING_WINDOWS = {'7일': 7*24, '30일': 30*24}  # 시간 단위

for asset_a, asset_b in INTEREST_PAIRS:
    if asset_a not in returns_df.columns or asset_b not in returns_df.columns:
        continue

    fig = go.Figure()

    for label, window in ROLLING_WINDOWS.items():
        if len(returns_df) < window:
            continue
        rolling_corr = returns_df[asset_a].rolling(window).corr(returns_df[asset_b])
        fig.add_trace(go.Scatter(x=rolling_corr.index, y=rolling_corr,
                                 name=f'롤링 {label}'))

    # 전체 기간 상관계수 기준선
    overall_corr = corr_matrix.loc[asset_a, asset_b]
    fig.add_hline(y=overall_corr, line_dash='dash', line_color='gray',
                  annotation_text=f'전체 기간: {overall_corr:.3f}')
    fig.add_hline(y=0, line_dash='dot', line_color='lightgray')

    fig.update_layout(
        title=f'{asset_a} vs {asset_b} — 롤링 상관관계',
        xaxis_title='시간', yaxis_title='상관계수',
        yaxis=dict(range=[-1, 1]),
        height=400, hovermode='x unified'
    )
    fig.show()

## 4. 수익률 산점도 행렬

In [ ]:
# 산점도 행렬 (pairplot)
# 자산 수가 많으면 상위 4개만
plot_cols = returns_df.columns[:min(6, len(returns_df.columns))]

g = sns.pairplot(returns_df[plot_cols], kind='scatter', diag_kind='kde',
                 plot_kws={'alpha': 0.3, 's': 10},
                 height=2.5)
g.fig.suptitle('수익률 산점도 행렬', y=1.02, fontsize=14)
plt.show()

In [ ]:
# 주요 쌍에 대한 상세 산점도 + 회귀선
for asset_a, asset_b in INTEREST_PAIRS[:3]:
    if asset_a not in returns_df.columns or asset_b not in returns_df.columns:
        continue

    x = returns_df[asset_a].values
    y = returns_df[asset_b].values

    # 회귀
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(x, y, alpha=0.3, s=15)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2,
            label=f'y = {slope:.4f}x + {intercept:.6f}\nR² = {r_value**2:.4f}, p = {p_value:.2e}')
    ax.set_xlabel(f'{asset_a} 수익률')
    ax.set_ylabel(f'{asset_b} 수익률')
    ax.set_title(f'{asset_a} vs {asset_b} — 수익률 회귀')
    ax.legend()
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()

## 5. 정규화 가격 비교 (Growth of $1)

In [ ]:
# 모든 자산을 1.0 기준으로 정규화
normalized = prices_df / prices_df.iloc[0]

fig = go.Figure()
for col in normalized.columns:
    fig.add_trace(go.Scatter(x=normalized.index, y=normalized[col], name=col))

fig.add_hline(y=1.0, line_dash='dash', line_color='gray')
fig.update_layout(
    title='정규화 가격 비교 (기준: 1.0)',
    xaxis_title='시간',
    yaxis_title='정규화 가격 (시작 = 1.0)',
    height=500, hovermode='x unified'
)
fig.show()